# Phase 0 pilot: orchestrated logging + fold-over ground truth

Runs the Phase 0 pilot (README's "Phase 0 split" section): a mix of natural-retrieval
logging episodes and fold-over ground-truth pairs for the 5 pre-registered memories,
resumable across Kaggle sessions via a Kaggle Dataset checkpoint.

**Built for Kaggle's "Save & Run All"**: every cell below runs unconditionally, in
order, with no manual choices -- the only thing to edit is `DATASET_SLUG` in the
settings cell (once, to your own Kaggle username/dataset), before the first run.

## Settings to click before running anything

- **Accelerator: GPU T4 x2** (Notebook settings, right sidebar) -- both GPUs are used by default (`WORKERS = 2`); run `multi_gpu_sanity_check.ipynb` once first to confirm both are actually being used before committing to a full session.
- **Internet: ON** (same sidebar) -- needed for pip installs and downloads.
- **Kaggle API token**: open **Add-ons -> Secrets**, add a secret labeled exactly `KAGGLE_API_TOKEN` (format `KGAT_...`, from kaggle.com Settings -> Account -> API tokens), and toggle it on for this notebook -- required for the checkpoint Dataset push/pull below.
- **Add Input -> your checkpoint Dataset**: on the very first run this dataset won't
  exist yet, so there's nothing to attach -- `run_phase0_pilot.py` detects that and
  starts fresh, printing "assuming this is the FIRST session." After the first run
  completes (it creates the dataset via the `kaggle` CLI automatically), attach that
  same dataset as an input on every later session so it can resume.
- **Time budget**: the run cell below uses a 10.5-hour budget by default, under
  Kaggle's 12-hour session cap -- edit `TIME_BUDGET_SECONDS` if you want a shorter
  session (e.g. for a quick test).
- **Checkpoint interval**: pushes an updated checkpoint every 2 hours by default
  (`CHECKPOINT_INTERVAL_SECONDS`), not just at the end -- a crash loses at most about
  that much work, not the whole session.

## What happens each session

1. Copies in the last checkpoint (if any).
2. Generates the job list on the very first session only (same file reused after).
3. Runs jobs (logging + ground-truth fold pairs) until the time budget or the job
   list is exhausted, whichever comes first.
4. Writes the pre-registered validation metrics against whatever's done so far.
5. Pushes the updated checkpoint back out as a new Dataset version.

Run this notebook (Save & Run All) once per Kaggle session until the job list is
exhausted -- check the last cell's printed summary to see how much is left.

## Settings -- edit DATASET_SLUG once, before the first run

In [ ]:
DATASET_SLUG = "your-kaggle-username/memory-pilot-phase0-checkpoints"  # EDIT THIS
N_LOGGING = 1050       # README's "Phase 0 split" section: ~70% of ~1500 episodes
N_GT_PAIRS = 225       # ~30% of ~1500 episodes, as fold-over pairs (2 episodes/pair)
TIME_BUDGET_SECONDS = 37_800  # 10.5h, under Kaggle's 12h session cap (time guard)
CHECKPOINT_INTERVAL_SECONDS = 7_200  # push to the checkpoint Dataset every ~2h while running, not just at the end
WORKERS = 2  # one per GPU (T4 x2) -- run multi_gpu_sanity_check.ipynb once before the first real session


In [ ]:
if 'your-kaggle-username' in DATASET_SLUG:
    raise SystemExit(
        'FAIL: DATASET_SLUG is still the placeholder value -- edit the settings cell above '
        "to your own '<kaggle-username>/<dataset-name>' before running this notebook."
    )
print('DATASET_SLUG looks real:', DATASET_SLUG)


## 0. Setup

In [ ]:
import os
# Standard fix for the PyTorch CUDA caching allocator's OOM-from-fragmentation
# failure mode -- set before anything imports torch. See README.md's OOM section.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
print('GPU:')
subprocess.run(['nvidia-smi'])


In [ ]:
!git clone https://github.com/ibo-agentic/memory-pilot.git
%cd memory-pilot
!git rev-parse HEAD


In [ ]:
!pip install alfworld tqdm termcolor sentence-transformers --quiet
!pip install transformers accelerate bitsandbytes --quiet
!alfworld-download

# Editable-installs both packages so alfworld_pilot and memory_ope import in
# EVERY process (including run_jobs.py's own subprocess workers) with no
# PYTHONPATH needed anywhere -- verified locally, see README's Orchestration section.
!pip install -e . --quiet
!pip install -e alfworld_pilot --quiet
!pip install --upgrade kaggle --quiet


In [ ]:
import os

# Kaggle's newer single-token API auth (KAGGLE_API_TOKEN, format KGAT_...)
# replaces the old kaggle.json username/key pair. Read it from Kaggle
# Secrets -- never hardcode it, never print it.
try:
    from kaggle_secrets import UserSecretsClient
    _kaggle_api_token = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
except Exception as e:
    raise SystemExit(
        'FAIL: could not read the KAGGLE_API_TOKEN secret -- add it via this '
        'notebook\'s Add-ons -> Secrets menu (label exactly KAGGLE_API_TOKEN, '
        f'value starting KGAT_...) before running this cell. Error: {e}'
    )

os.environ['KAGGLE_API_TOKEN'] = _kaggle_api_token
del _kaggle_api_token  # don't leave the token sitting in a notebook-visible variable
print('KAGGLE_API_TOKEN loaded from Kaggle Secrets and set as an environment variable (value not printed).')


In [ ]:
import sys, site, pathlib, importlib

# A Kaggle kernel's sys.path is snapshotted at process start, so a `pip
# install -e` a few cells ago is not guaranteed to be visible yet, and any
# earlier failed/partial import attempt can leave a broken module cached in
# sys.modules -- both observed for real on Kaggle (session 1 needed this
# fixed manually). Re-scan site-packages for new .pth files (this is what
# actually registers an editable install's path), and drop any already-
# cached alfworld_pilot/memory_ope modules so the next import is fresh.
importlib.reload(site)
site_dirs = list(site.getsitepackages())
try:
    site_dirs.append(site.getusersitepackages())
except Exception:
    pass
for _dir in site_dirs:
    if _dir and pathlib.Path(_dir).is_dir():
        site.addsitedir(_dir)

for _mod_name in list(sys.modules):
    if _mod_name == 'alfworld_pilot' or _mod_name.startswith('alfworld_pilot.') or _mod_name == 'memory_ope' or _mod_name.startswith('memory_ope.'):
        del sys.modules[_mod_name]

print('sys.path refreshed and cached alfworld_pilot/memory_ope modules cleared.')


In [ ]:
try:
    import alfworld_pilot
    import memory_ope
    from alfworld_pilot import job_list, run_jobs, run_estimators, run_phase0_pilot  # noqa: F401
    print('Import check PASSED:')
    print('  alfworld_pilot ->', alfworld_pilot.__file__)
    print('  memory_ope     ->', memory_ope.__file__)
except ImportError as e:
    raise SystemExit(f'Import check FAILED: {e!r} -- fix the install cell above before proceeding.')


## 1. Run the pilot for this session

Idempotent and resumable: safe to run this exact cell again in a brand-new session
(after re-attaching the checkpoint Dataset as input) -- it picks up exactly where the
last session's Dataset version left off, per job_id, never re-running or skipping a
job incorrectly (see `tests/test_job_list_and_runner.py` and the handoff test).

In [ ]:
%cd alfworld_pilot
import sys
sys.argv = [
    'run_phase0_pilot',
    '--dataset-slug', DATASET_SLUG,
    '--n-logging', str(N_LOGGING),
    '--n-gt-pairs', str(N_GT_PAIRS),
    '--time-budget-seconds', str(TIME_BUDGET_SECONDS),
    '--checkpoint-interval-seconds', str(CHECKPOINT_INTERVAL_SECONDS),
    '--workers', str(WORKERS),
]
from alfworld_pilot import run_phase0_pilot
run_phase0_pilot.main()


## 2. Session summary

In [ ]:
import json, pathlib

results_path = pathlib.Path('/kaggle/working/logs_kaggle/phase0_results.jsonl')
job_list_path = pathlib.Path('/kaggle/working/logs_kaggle/phase0_job_list.json')

if job_list_path.exists():
    jobs = json.load(open(job_list_path))
    n_total = len(jobs)
else:
    n_total = 0

if results_path.exists():
    with open(results_path) as f:
        n_done = sum(1 for line in f if line.strip())
else:
    n_done = 0

print(f'Progress: {n_done}/{n_total} jobs done ({100 * n_done / n_total:.1f}%)' if n_total else 'No job list found.')
if n_total and n_done < n_total:
    print('Not finished -- attach the checkpoint Dataset this session just pushed as input on the next session, and Save & Run All again.')
elif n_total:
    print('All jobs done. See phase0_validation.json (also pushed to the checkpoint Dataset) for the validation metrics.')
